# Mini-TP 4 — Puntúa tu modelo sobre un flujo (starter)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA · Entrega individual, esta semana**

Aplica **tu modelo** (el de la Sesión 1) sobre un **flujo de eventos** en lugar de pedidos sueltos. Completa las celdas marcadas con `# TODO`.

**Qué debes entregar**
1. Un **flujo** de eventos con las *features* de tu modelo (simulado, o consumido de Kafka).
2. Un **consumidor** que puntúe tu modelo sobre **cada** evento (inferencia online).
3. **Métricas por ventana:** throughput (ev/s), latencia **p95** y un indicador simple de **drift** de entradas.
4. Una **alerta** cuando una métrica cruce un umbral.
5. Una **comparación** contra puntuar el mismo lote en **batch**, con una breve reflexión.

**Se evalúa:** que corra de punta a punta; inferencia online sobre el flujo; métricas por ventana correctas; y la reflexión streaming vs batch.

> Apóyate en `streaming_tutorial.ipynb`. Entorno uv: `uv add scikit-learn joblib numpy kafka-python`.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "joblib", "numpy"])
print("Listo.")

## 1. Tu modelo

Carga **tu** modelo. Si aún no lo tienes a mano, el placeholder deja probar la mecánica.

In [ ]:
import joblib, numpy as np
# TODO: carga tu modelo (debe exponer .predict(X))
# modelo = joblib.load("model.pkl")

# --- placeholder (reemplázalo por tu modelo) ---
from sklearn.linear_model import LogisticRegression
_X = np.random.RandomState(0).randn(300, 3); _y = (_X.sum(1) > 0).astype(int)
modelo = LogisticRegression().fit(_X, _y)
N_FEATURES = 3   # TODO: ajusta al número de features de tu modelo
print("Modelo cargado. n_features =", N_FEATURES)

## 2. El flujo

Simula un flujo con las features de tu modelo. Introduce un cambio de distribución a mitad de camino para poder detectar drift. (Alternativa: consumir de Kafka como en el tutorial.)

In [ ]:
import queue, threading, time, random
stream = queue.Queue(); STOP = object()
N = 800

def productor():
    for i in range(N):
        # TODO: genera un vector de N_FEATURES acorde a tu dominio
        mu = 0.0 if i < N//2 else 1.2
        stream.put({"values": [random.gauss(mu, 1) for _ in range(N_FEATURES)], "t": time.time()})
        time.sleep(0.0005)
    stream.put(STOP)
print("Productor listo.")

## 3. Consumidor con inferencia online + métricas por ventana

In [ ]:
from collections import deque
import numpy as np

latencias = []; ventana = deque(); alertas = []

def consumidor():
    procesados = 0
    while True:
        ev = stream.get()
        if ev is STOP: break
        t0 = time.perf_counter()
        pred = modelo.predict([ev["values"]])[0]           # inferencia online
        latencias.append((time.perf_counter() - t0) * 1000)

        ventana.append((ev["t"], ev["values"][0], pred))
        ahora = ev["t"]
        while ventana and ahora - ventana[0][0] > 1.0:      # ventana deslizante 1s
            ventana.popleft()

        procesados += 1
        if procesados % 200 == 0:
            media_f0 = float(np.mean([w[1] for w in ventana]))
            # TODO: define tu indicador de drift y tu umbral de alerta
            if media_f0 > 0.6:
                alertas.append((procesados, media_f0))

t0 = time.time()
pt = threading.Thread(target=productor); ct = threading.Thread(target=consumidor)
pt.start(); ct.start(); pt.join(); ct.join()
dur = time.time() - t0

print(f"throughput : {len(latencias)/dur:.0f} ev/s")
print(f"p95        : {np.percentile(latencias, 95):.3f} ms")
print(f"alertas de drift: {alertas}")

## 4. Comparación contra batch

Puntúa el mismo conjunto de eventos de una sola vez (batch) y compara tiempo/uso frente al streaming.

In [ ]:
# TODO: arma un lote con los mismos vectores y puntúalo de una vez con modelo.predict(LOTE)
# batch_pred = modelo.predict(LOTE)
# Compara: ¿latencia por evento? ¿frescura del resultado? ¿cuándo conviene cada uno?
print("Pendiente: implementar el scoring batch y comparar.")

### Reflexión (completa)

_TODO: 3–5 líneas. ¿Qué te dio el streaming que el batch no? ¿Cuándo usarías cada uno en tu plataforma? ¿Cómo detectaste el drift y qué harías al dispararse la alerta (reentrenar, avisar, degradar)?_

## 5. (Opcional) Con Kafka en Docker

Levanta un broker y reemplaza la cola por un topic real:
```bash
docker run -d --name redpanda -p 9092:9092 \
  redpandadata/redpanda redpanda start --overprovisioned --smp 1 --check=false
uv add kafka-python
```
Produce a un topic `eventos` y consume puntuando tu modelo (ver la parte 6 del tutorial).